In [ ]:
# === Cell 1: Data Root ===
import os

# Data root; override with the KI_SORT_ROOT environment variable.
KI_SORT_ROOT = os.environ.get("KI_SORT_ROOT", "/VOL_D/fio.h")


In [ ]:
# === Cell 2: Imports ===

from __future__ import annotations

from pathlib import Path
from datetime import datetime
import re
import warnings

import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_fscore_support,
)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print(f"numpy  : {np.__version__}")
print(f"pandas : {pd.__version__}")
import sklearn
print(f"sklearn: {sklearn.__version__}")


In [ ]:
# === Cell 3: Config ===

# --- Output tree ---
OUTPUT_ROOT  = Path(f"{KI_SORT_ROOT}/KI_Sort_Project/08_Generalizability/20260701_Classifier_Generalizability_Assessment_independent_capture_session")
PROJECT_NAME = OUTPUT_ROOT.name
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# False blocks overwriting existing output CSV and PNG files.
ALLOW_OVERWRITE = True

# --- Source trees ---
DATASET_ROOT   = Path(f"{KI_SORT_ROOT}/KI_Sort_Datasets/Masterthesis_TW_Dataset")
BENCHMARK_ROOT = Path(f"{KI_SORT_ROOT}/KI_Sort_Project/05_IR_Classification/20260615_Classifier_Benchmark_Preprocessing_Influence")

CLASSIFIER_NAME = "LR"

# --- Part 1: preprocessing tiers ---
# Each tier pairs the independent-capture-session spectra preprocessed at that tier
# with the benchmark best estimator of the same tier. These dirs contain no LDPE files.
TIER_RUNS = {
    "C0": {
        "display":    "C0 Cut Spectra",
        "input_dir":  DATASET_ROOT / "resampled_spectra_20260702_063954_wo_ldpe",
        "model_path": BENCHMARK_ROOT / "C0_cut_spectra/results/models/C0_cut_spectra_LR_best_estimator.joblib",
    },
    "C1": {
        "display":    "C1 SNV",
        "input_dir":  DATASET_ROOT / "norm_only_spectra_20260702_064135_wo_ldpe",
        "model_path": BENCHMARK_ROOT / "C1_z-score/results/models/C1_z-score_LR_best_estimator.joblib",
    },
    "C2": {
        "display":    "C2 ALS + SNV",
        "input_dir":  DATASET_ROOT / "norm_bc_spectra_20260702_064428_wo_ldpe",
        "model_path": BENCHMARK_ROOT / "C2_als_z-score/results/models/C2_als_z-score_LR_best_estimator.joblib",
    },
}
TIER_ORDER = ["C0", "C1", "C2"]

# --- Part 2: deployment classifier ---
DEPLOYMENT_ROOT = Path(f"{KI_SORT_ROOT}/KI_Sort_Project/05_IR_Classification/20260629_LR_Deployment_Training")

# Explicit path to the deployment joblib; None resolves it by glob under DEPLOYMENT_ROOT.
DEPLOYMENT_MODEL_PATH = None

# The deployment LR is fit on C2-preprocessed spectra (ALS baseline + z-score), so the
# input is the C2 product, including LDPE.
DEPLOYMENT_INPUT_DIR = DATASET_ROOT / "norm_bc_spectra_20260702_064428"

DEPLOY_RUNS = {
    "DEPLOY_NO_LDPE": {
        "display":          "Deployment LR, LDPE Excluded",
        "exclude_prefixes": ("LDPE",),
        "split_ldpe":       False,
    },
    "DEPLOY_LDPE_SPLIT": {
        "display":          "Deployment LR, LDPE Carried Separately",
        "exclude_prefixes": (),
        "split_ldpe":       True,
    },
}
DEPLOY_ORDER = ["DEPLOY_NO_LDPE", "DEPLOY_LDPE_SPLIT"]

# --- Ground-truth derivation ---
# Merged: HDPE and LDPE map onto the single PE class of the models.
PREFIX_TO_CLASS_MERGED = {"HDPE": "PE", "LDPE": "PE", "PET": "PET", "PP": "PP", "PS": "PS"}
# Split: HDPE and LDPE stay distinct on the true axis; neither exists on the
# predicted axis, so this run reports a confusion matrix only, no F1 / Brier.
PREFIX_TO_CLASS_SPLIT  = {"HDPE": "HDPE", "LDPE": "LDPE", "PET": "PET", "PP": "PP", "PS": "PS"}

# True classes that a PE prediction resolves correctly under the split mapping.
PE_EQUIVALENT_TRUE = ("HDPE", "LDPE")

# --- Class axes ---
TIER_CLASSES        = ["PE", "PET", "PP", "PS"]                    # Part 1, square matrix
CANONICAL_PRED_ORDER = ["PE", "PET", "PP", "PS", "Paper"]           # Part 2 predicted axis
DEPLOY_ROW_MERGED   = ["PE", "PET", "PP", "PS"]
DEPLOY_ROW_SPLIT    = ["HDPE", "LDPE", "PET", "PP", "PS"]

# --- Spectrum file format ---
SPECTRUM_SEP = ";"

# --- Plot style ---
_FW        = 16 / 2.54
_FONT_SIZE = 10
_BAND_COLOR    = "#f0f0f0"
_YLIM_HEADROOM = 1.18
_HEADROOM_LEGEND = 1.60          # extra headroom on panels carrying an in-axes legend
_F1_COLOR    = "#4c72b0"
_BRIER_COLOR = "#c44e52"
TIER_COLORS  = {"C0": "#1f77b4", "C1": "#ff7f0e", "C2": "#2ca02c"}

print(f"PROJECT_NAME         : {PROJECT_NAME}")
print(f"OUTPUT_ROOT          : {OUTPUT_ROOT}")
print(f"DEPLOYMENT_INPUT_DIR : {DEPLOYMENT_INPUT_DIR.name}")
print(f"ALLOW_OVERWRITE      : {ALLOW_OVERWRITE}")


In [ ]:
# === Cell 4: Helper Functions ===

_PREFIX_RE = re.compile(r"^(?P<prefix>[A-Za-z0-9]+)_")


def derive_prefix(name: str):
    """Leading filename token, upper-cased, or None."""
    m = _PREFIX_RE.match(name)
    return m.group("prefix").upper() if m else None


def build_gt_table(input_dir: Path, prefix_map: dict, exclude_prefixes=()) -> pd.DataFrame:
    """Discover spectrum files and attach a true class from the filename prefix."""
    if not input_dir.is_dir():
        raise FileNotFoundError(f"Input dir does not exist: {input_dir}")

    excl = {p.upper() for p in exclude_prefixes}
    all_paths = sorted(p for p in input_dir.iterdir() if p.is_file())

    rows, unmapped, excluded = [], [], []
    for p in all_paths:
        prefix = derive_prefix(p.name)
        if prefix is None or prefix not in prefix_map:
            unmapped.append(p.name)
            continue
        if prefix in excl:
            excluded.append(p.name)
            continue
        rows.append({
            "filename":   p.name,
            "filepath":   str(p),
            "prefix":     prefix,
            "true_class": prefix_map[prefix],
        })

    gt = pd.DataFrame(rows).sort_values("filename").reset_index(drop=True)
    print(f"  files found : {len(all_paths)} | mapped: {len(gt)} | "
          f"excluded: {len(excluded)} | unmapped: {len(unmapped)}")
    if unmapped:
        preview = ", ".join(unmapped[:8]) + (" ..." if len(unmapped) > 8 else "")
        print(f"  unmapped    : {preview}")
    return gt


def read_spectrum_file(path: Path) -> pd.DataFrame:
    """Load a headerless, semicolon-delimited `wavenumber;reflectance` spectrum."""
    df = pd.read_csv(path, sep=SPECTRUM_SEP, header=None, engine="python")
    if df.shape[0] == 0:
        raise ValueError(f"{path.name}: empty file")
    if df.shape[1] != 2:
        raise ValueError(f"{path.name}: expected 2 columns, got {df.shape[1]}")
    df = df.apply(pd.to_numeric, errors="coerce")
    if df.isna().any().any():
        raise ValueError(f"{path.name}: contains non-numeric / NaN values")
    df.columns = ["wavenumber", "reflectance"]
    return df.sort_values("wavenumber").reset_index(drop=True)


def build_feature_matrix(gt: pd.DataFrame) -> np.ndarray:
    """Stack reflectance vectors in gt row order; enforce a common axis length."""
    feats, axis_ref = [], None
    for _, row in gt.iterrows():
        spec = read_spectrum_file(Path(row["filepath"]))
        axis = spec["wavenumber"].to_numpy()
        if axis_ref is None:
            axis_ref = axis
        elif len(axis) != len(axis_ref):
            raise ValueError(
                f"{row['filename']}: feature length {len(axis)} != reference {len(axis_ref)}"
            )
        feats.append(spec["reflectance"].to_numpy())
    return np.vstack(feats)


def _patch_missing_multi_class(model):
    """Restore LogisticRegression.multi_class, which sklearn >= 1.8.0 no longer defines."""
    from sklearn.linear_model import LogisticRegression
    steps = getattr(model, "named_steps", None)
    targets = list(steps.values()) if steps else [model]
    for t in targets:
        if isinstance(t, LogisticRegression) and not hasattr(t, "multi_class"):
            t.multi_class = "deprecated"
    return model


def load_model(model_path: Path):
    """Load a joblib model and return it with its class list."""
    if not model_path.is_file():
        raise FileNotFoundError(f"Model not found: {model_path}")
    model = _patch_missing_multi_class(joblib.load(model_path))
    return model, list(model.classes_)


def resolve_deployment_model(explicit, root: Path) -> Path:
    """Return the configured deployment joblib, or glob-resolve it under root."""
    if explicit is not None:
        return Path(explicit)
    hits = sorted(root.rglob("*_deployment.joblib"))
    if not hits:
        raise FileNotFoundError(f"No *_deployment.joblib found under {root}")
    if len(hits) > 1:
        print(f"  multiple deployment joblibs found, using the first:")
        for h in hits:
            print(f"    {h}")
    return hits[0]


def predict_table(model, model_classes, gt: pd.DataFrame, X: np.ndarray,
                  tag: str, model_path: Path, run_ts: str) -> pd.DataFrame:
    """Run predict / predict_proba and assemble the standard results table."""
    n_expected = getattr(model, "n_features_in_", X.shape[1])
    if X.shape[1] != n_expected:
        raise ValueError(
            f"[{tag}] feature length {X.shape[1]} != model n_features_in_ {n_expected}. "
            "Input spectra are not on the same axis as the training data."
        )

    unknown = sorted(set(gt["true_class"]) - set(model_classes))
    if unknown:
        warnings.warn(f"[{tag}] GT classes absent from model.classes_: {unknown}")

    y_pred = model.predict(X)
    proba  = model.predict_proba(X)

    out = pd.DataFrame({
        "filename":   gt["filename"].to_numpy(),
        "prefix":     gt["prefix"].to_numpy(),
        "true_class": gt["true_class"].to_numpy(),
        "pred_class": y_pred,
    })
    for i, cls in enumerate(model_classes):
        out[f"proba_{cls}"] = proba[:, i]
    out["run_tag"]         = tag
    out["classifier_name"] = CLASSIFIER_NAME
    out["run_timestamp"]   = run_ts
    out["model_path"]      = str(model_path)
    return out


def rect_confusion(y_true, y_pred, row_labels, col_labels):
    """Confusion counts on independent true / predicted axes."""
    cm = pd.DataFrame(0, index=list(row_labels), columns=list(col_labels), dtype=int)
    off_axis = []
    for t, p in zip(y_true, y_pred):
        if t in cm.index and p in cm.columns:
            cm.loc[t, p] += 1
        else:
            off_axis.append((t, p))
    cm.index.name   = "true_class"
    cm.columns.name = "pred_class"
    return cm, off_axis


def per_class_brier(df: pd.DataFrame, classes) -> dict:
    """One-vs-rest Brier score per class (not the multiclass Brier score of the benchmark)."""
    out = {}
    for c in classes:
        col = f"proba_{c}"
        if col not in df.columns:
            continue
        y_bin = (df["true_class"].to_numpy() == c).astype(float)
        p = df[col].to_numpy()
        out[c] = float(np.mean((p - y_bin) ** 2))
    return out


def pred_axis(model_classes) -> list:
    """Canonical predicted-class order, extended with anything unexpected."""
    known = [c for c in CANONICAL_PRED_ORDER if c in model_classes]
    extra = [c for c in model_classes if c not in known]
    return known + extra


def latest_csv(stem_fragment: str):
    """Return the output CSV for a stem fragment, or None if it does not exist yet."""
    out = OUTPUT_ROOT / f"{PROJECT_NAME}_{stem_fragment}.csv"
    return out if out.is_file() else None


def save_csv(df: pd.DataFrame, stem: str, index=False) -> Path:
    """Save a CSV under a fixed filename; ALLOW_OVERWRITE=False blocks overwriting."""
    out = OUTPUT_ROOT / f"{PROJECT_NAME}_{stem}.csv"
    if out.exists() and not ALLOW_OVERWRITE:
        raise RuntimeError(f"Refusing to overwrite {out.name} with ALLOW_OVERWRITE=False")
    df.to_csv(out, index=index)
    print(f"  saved: {out.name}")
    return out


def save_fig(fig, stem: str) -> Path:
    out = OUTPUT_ROOT / f"{PROJECT_NAME}_{stem}.png"
    if out.exists() and not ALLOW_OVERWRITE:
        raise RuntimeError(f"Refusing to overwrite {out.name} with ALLOW_OVERWRITE=False")
    fig.savefig(out, dpi=150, bbox_inches="tight")
    print(f"  saved: {out.name}")
    return out


def _apply_rcparams():
    plt.rcParams["font.family"]     = "DejaVu Sans"
    plt.rcParams["font.size"]       = _FONT_SIZE
    plt.rcParams["axes.titlesize"]  = _FONT_SIZE + 1
    plt.rcParams["axes.labelsize"]  = _FONT_SIZE
    plt.rcParams["legend.fontsize"] = _FONT_SIZE - 1
    plt.rcParams["savefig.dpi"]     = 150


def _add_bands(ax, n):
    for i in range(n):
        if i % 2 == 0:
            ax.axvspan(i - 0.5, i + 0.5, color=_BAND_COLOR, zorder=0)


def _safe_top(values, headroom=_YLIM_HEADROOM):
    """Upper y-limit with headroom; falls back to 1.0 for an all-zero series."""
    vals = [v for v in values if np.isfinite(v)]
    top = max(vals) if vals and max(vals) > 0 else 1.0
    return top * headroom


def _panel_label(ax, letter, title):
    ax.set_title(f"$\\bf{{({letter})}}$ {title}")


def _draw_cm(ax, cm: pd.DataFrame, fmt="d", cmap="Blues", annot=None):
    """Render a rectangular confusion matrix with in-cell annotations."""
    vals = cm.to_numpy()
    ax.imshow(vals, cmap=cmap, aspect="auto")
    ax.set_xticks(range(cm.shape[1]))
    ax.set_xticklabels(list(cm.columns), rotation=45, ha="right")
    ax.set_yticks(range(cm.shape[0]))
    ax.set_yticklabels(list(cm.index))
    thr = vals.max() / 2 if vals.max() else 0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            txt = annot[i][j] if annot is not None else format(vals[i, j], fmt)
            ax.text(j, i, txt, ha="center", va="center",
                    fontsize=_FONT_SIZE - 2,
                    color="white" if vals[i, j] > thr else "black")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")


print("Helpers defined.")


In [ ]:
# === Cell 5: Run Preprocessing Tier Batch ===

_run_ts = datetime.now().strftime("%Y%m%d_%H%M%S")
print(f"run timestamp: {_run_ts}\n")

_tier_metric_rows = []

for _tier in TIER_ORDER:
    _cfg = TIER_RUNS[_tier]
    print(f"[{_tier}] {_cfg['display']}")
    print(f"  input : {_cfg['input_dir'].name}")
    print(f"  model : {_cfg['model_path'].name}")

    _gt = build_gt_table(_cfg["input_dir"], PREFIX_TO_CLASS_MERGED)
    _model, _model_classes = load_model(_cfg["model_path"])
    print(f"  model classes: {_model_classes}")

    _X = build_feature_matrix(_gt)
    print(f"  feature matrix: {_X.shape}")

    _res = predict_table(_model, _model_classes, _gt, _X,
                         _tier, _cfg["model_path"], _run_ts)
    save_csv(_res, f"predictions_{_tier}")

    # Square confusion matrix over the four polymer classes.
    _cm, _off = rect_confusion(_res["true_class"], _res["pred_class"],
                               TIER_CLASSES, TIER_CLASSES)
    if _off:
        _off_pred = sorted({p for _, p in _off})
        warnings.warn(f"[{_tier}] {len(_off)} predictions outside {TIER_CLASSES}: {_off_pred}")
    save_csv(_cm, f"confusion_{_tier}", index=True)

    _prec, _rec, _f1, _sup = precision_recall_fscore_support(
        _res["true_class"], _res["pred_class"],
        labels=TIER_CLASSES, average=None, zero_division=0
    )
    _brier = per_class_brier(_res, TIER_CLASSES)

    for _i, _c in enumerate(TIER_CLASSES):
        _tier_metric_rows.append({
            "tier": _tier, "tier_display": _cfg["display"], "scope": "per_class",
            "class": _c, "support": int(_sup[_i]),
            "precision": float(_prec[_i]), "recall": float(_rec[_i]),
            "f1": float(_f1[_i]), "brier_ovr": _brier.get(_c, np.nan),
        })

    _acc = accuracy_score(_res["true_class"], _res["pred_class"])
    _macro_f1 = f1_score(_res["true_class"], _res["pred_class"],
                         labels=TIER_CLASSES, average="macro", zero_division=0)
    _mean_brier = float(np.mean([_brier[c] for c in TIER_CLASSES if c in _brier]))
    _tier_metric_rows.append({
        "tier": _tier, "tier_display": _cfg["display"], "scope": "overall",
        "class": "ALL", "support": int(len(_res)),
        "precision": np.nan, "recall": np.nan,
        "f1": float(_macro_f1), "brier_ovr": _mean_brier,
        "accuracy": float(_acc),
    })
    print(f"  accuracy: {_acc:.4f} | macro F1: {_macro_f1:.4f} | mean OVR Brier: {_mean_brier:.4f}\n")

tier_metrics = pd.DataFrame(_tier_metric_rows)
save_csv(tier_metrics, "tier_metrics")

print("\nOverall by tier:")
print(tier_metrics[tier_metrics["scope"] == "overall"]
      [["tier", "accuracy", "f1", "brier_ovr"]].to_string(index=False))


In [ ]:
# === Cell 6: Run Deployment Batch ===

_deploy_ts = datetime.now().strftime("%Y%m%d_%H%M%S")
print(f"run timestamp: {_deploy_ts}\n")

_deploy_model_path = resolve_deployment_model(DEPLOYMENT_MODEL_PATH, DEPLOYMENT_ROOT)
print(f"deployment model : {_deploy_model_path}")
print(f"deployment input : {DEPLOYMENT_INPUT_DIR}\n")

_deploy_model, _deploy_classes = load_model(_deploy_model_path)
_deploy_pred_axis = pred_axis(_deploy_classes)
print(f"model classes    : {_deploy_classes}")
print(f"predicted axis   : {_deploy_pred_axis}\n")

_deploy_summary_rows = []

for _tag in DEPLOY_ORDER:
    _cfg = DEPLOY_RUNS[_tag]
    print(f"[{_tag}] {_cfg['display']}")

    _pmap = PREFIX_TO_CLASS_SPLIT if _cfg["split_ldpe"] else PREFIX_TO_CLASS_MERGED
    _rows = DEPLOY_ROW_SPLIT if _cfg["split_ldpe"] else DEPLOY_ROW_MERGED

    _gt = build_gt_table(DEPLOYMENT_INPUT_DIR, _pmap,
                         exclude_prefixes=_cfg["exclude_prefixes"])
    print("  true class distribution:")
    print("   ", _gt["true_class"].value_counts().sort_index().to_dict())

    _X = build_feature_matrix(_gt)
    print(f"  feature matrix: {_X.shape}")

    _res = predict_table(_deploy_model, _deploy_classes, _gt, _X,
                         _tag, _deploy_model_path, _deploy_ts)
    save_csv(_res, f"predictions_{_tag}")

    # The predicted axis always includes Paper so Paper predictions are counted.
    _cm, _off = rect_confusion(_res["true_class"], _res["pred_class"],
                               _rows, _deploy_pred_axis)
    if _off:
        warnings.warn(f"[{_tag}] {len(_off)} rows fell outside the configured axes")
    save_csv(_cm, f"confusion_{_tag}", index=True)

    if _cfg["split_ldpe"]:
        # HDPE and LDPE count as correct when predicted as PE.
        _resolved = np.where(
            np.isin(_res["true_class"], PE_EQUIVALENT_TRUE),
            (_res["pred_class"] == "PE"),
            (_res["pred_class"] == _res["true_class"]),
        )
        _acc = float(np.mean(_resolved))
        _macro_f1 = np.nan
        _mean_brier = np.nan
        print(f"  PE-resolved accuracy: {_acc:.4f}")
        for _c in _rows:
            _m = _res["true_class"] == _c
            if not _m.any():
                continue
            _dist = _res.loc[_m, "pred_class"].value_counts(normalize=True)
            _top = ", ".join(f"{k}={v:.1%}" for k, v in _dist.items())
            print(f"    {_c:<5} n={int(_m.sum()):<4} -> {_top}")
    else:
        _acc = accuracy_score(_res["true_class"], _res["pred_class"])
        _macro_f1 = f1_score(_res["true_class"], _res["pred_class"],
                             labels=_rows, average="macro", zero_division=0)
        _brier = per_class_brier(_res, _rows)
        _mean_brier = float(np.mean([_brier[c] for c in _rows if c in _brier]))
        print(f"  accuracy: {_acc:.4f} | macro F1: {_macro_f1:.4f} | "
              f"mean OVR Brier: {_mean_brier:.4f}")

    _n_paper = int((_res["pred_class"] == "Paper").sum())
    print(f"  predictions landing on Paper: {_n_paper}\n")

    _deploy_summary_rows.append({
        "run_tag": _tag, "display": _cfg["display"],
        "n_spectra": int(len(_res)),
        "accuracy": _acc, "macro_f1": _macro_f1, "mean_brier_ovr": _mean_brier,
        "n_pred_paper": _n_paper,
        "input_dir": str(DEPLOYMENT_INPUT_DIR),
        "model_path": str(_deploy_model_path),
        "run_timestamp": _deploy_ts,
    })

deploy_summary = pd.DataFrame(_deploy_summary_rows)
save_csv(deploy_summary, "deployment_summary")


In [ ]:
# === Cell 7: Plot Per-Tier Panels ===

_apply_rcparams()

for _tier in TIER_ORDER:
    _pred_csv = latest_csv(f"predictions_{_tier}")
    if _pred_csv is None:
        print(f"[{_tier}] no prediction CSV found, skipping")
        continue

    _df = pd.read_csv(_pred_csv)
    _display = TIER_RUNS[_tier]["display"]
    _n = len(TIER_CLASSES)

    _cm, _ = rect_confusion(_df["true_class"], _df["pred_class"],
                            TIER_CLASSES, TIER_CLASSES)
    _, _, _f1, _ = precision_recall_fscore_support(
        _df["true_class"], _df["pred_class"],
        labels=TIER_CLASSES, average=None, zero_division=0
    )
    _brier = per_class_brier(_df, TIER_CLASSES)
    _brier_vals = [_brier[c] for c in TIER_CLASSES]

    _fig, _axes = plt.subplots(1, 3, figsize=(_FW, _FW * 0.36))

    _ax = _axes[0]
    _draw_cm(_ax, _cm)
    _panel_label(_ax, "a", "Confusion Matrix")
    _ax.set_box_aspect(1)

    _ax = _axes[1]
    _add_bands(_ax, _n)
    _ax.bar(range(_n), _f1, color=_F1_COLOR, zorder=2)
    _ax.set_xticks(range(_n)); _ax.set_xticklabels(TIER_CLASSES, rotation=45, ha="right")
    _ax.set_xlim(-0.5, _n - 0.5)
    _ax.set_ylim(0, _safe_top(_f1))
    _ax.set_ylabel("F1 Score")
    _panel_label(_ax, "b", "Per-Class F1 Score")
    _ax.set_box_aspect(1)

    _ax = _axes[2]
    _add_bands(_ax, _n)
    _ax.bar(range(_n), _brier_vals, color=_BRIER_COLOR, zorder=2)
    _ax.set_xticks(range(_n)); _ax.set_xticklabels(TIER_CLASSES, rotation=45, ha="right")
    _ax.set_xlim(-0.5, _n - 0.5)
    _ax.set_ylim(0, _safe_top(_brier_vals))
    _ax.set_ylabel("Brier Score")
    _panel_label(_ax, "c", "Per-Class Brier Score")
    _ax.set_box_aspect(1)

    _fig.suptitle(f"{CLASSIFIER_NAME} on Independent Capture Session — {_display}")
    _fig.tight_layout()
    save_fig(_fig, f"tier_panels_{_tier}")
    plt.show()


In [ ]:
# === Cell 8: Plot Cross-Tier Summary ===

_apply_rcparams()

_metrics_csv = latest_csv("tier_metrics")
if _metrics_csv is None:
    print("No tier metrics CSV found. Run the tier batch cell first.")
else:
    _m = pd.read_csv(_metrics_csv)
    _per = _m[_m["scope"] == "per_class"]
    _ovr = _m[_m["scope"] == "overall"].set_index("tier")

    _n = len(TIER_CLASSES)
    _nt = len(TIER_ORDER)
    _w = 0.8 / _nt
    _x = np.arange(_n)

    _fig, _axes = plt.subplots(2, 2, figsize=(_FW, _FW * 0.78))

    # (a) per-class F1 grouped by tier
    _ax = _axes[0, 0]
    _add_bands(_ax, _n)
    for _k, _tier in enumerate(TIER_ORDER):
        _sub = _per[_per["tier"] == _tier].set_index("class").reindex(TIER_CLASSES)
        _ax.bar(_x + (_k - (_nt - 1) / 2) * _w, _sub["f1"], _w,
                color=TIER_COLORS[_tier], label=_tier, zorder=2)
    _ax.set_xticks(_x); _ax.set_xticklabels(TIER_CLASSES)
    _ax.set_xlim(-0.5, _n - 0.5)
    _ax.set_ylim(0, _safe_top([_per["f1"].max()], _HEADROOM_LEGEND))
    _ax.set_ylabel("F1 Score")
    _ax.legend(ncols=1, loc="upper right")
    _panel_label(_ax, "a", "Per-Class F1 Score by Tier")

    # (b) per-class one-vs-rest Brier grouped by tier
    _ax = _axes[0, 1]
    _add_bands(_ax, _n)
    for _k, _tier in enumerate(TIER_ORDER):
        _sub = _per[_per["tier"] == _tier].set_index("class").reindex(TIER_CLASSES)
        _ax.bar(_x + (_k - (_nt - 1) / 2) * _w, _sub["brier_ovr"], _w,
                color=TIER_COLORS[_tier], label=_tier, zorder=2)
    _ax.set_xticks(_x); _ax.set_xticklabels(TIER_CLASSES)
    _ax.set_xlim(-0.5, _n - 0.5)
    _ax.set_ylim(0, _safe_top([_per["brier_ovr"].max()], _HEADROOM_LEGEND))
    _ax.set_ylabel("Brier Score")
    _ax.legend(ncols=1, loc="upper right")
    _panel_label(_ax, "b", "Per-Class Brier Score by Tier")

    # (c) accuracy and macro F1 by tier
    _ax = _axes[1, 0]
    _add_bands(_ax, _nt)
    _xt = np.arange(_nt)
    _acc = [_ovr.loc[t, "accuracy"] for t in TIER_ORDER]
    _mf1 = [_ovr.loc[t, "f1"] for t in TIER_ORDER]
    _ax.bar(_xt - 0.2, _acc, 0.4, color="#8c8c8c", label="Accuracy", zorder=2)
    _ax.bar(_xt + 0.2, _mf1, 0.4, color=_F1_COLOR, label="Macro F1", zorder=2)
    for _i in range(_nt):
        _ax.text(_xt[_i] - 0.2, _acc[_i], f"{_acc[_i]:.2f}", ha="center", va="bottom",
                 fontsize=_FONT_SIZE - 2, fontweight="bold")
        _ax.text(_xt[_i] + 0.2, _mf1[_i], f"{_mf1[_i]:.2f}", ha="center", va="bottom",
                 fontsize=_FONT_SIZE - 2, fontweight="bold")
    _ax.set_xticks(_xt); _ax.set_xticklabels(TIER_ORDER)
    _ax.set_xlim(-0.5, _nt - 0.5)
    _ax.set_ylim(0, _safe_top(_acc + _mf1, _HEADROOM_LEGEND))
    _ax.set_ylabel("Score")
    _ax.legend(ncols=1, loc="upper right")
    _panel_label(_ax, "c", "Overall Performance by Tier")

    # (d) mean one-vs-rest Brier by tier
    _ax = _axes[1, 1]
    _add_bands(_ax, _nt)
    _mb = [_ovr.loc[t, "brier_ovr"] for t in TIER_ORDER]
    _ax.bar(_xt, _mb, 0.6, color=_BRIER_COLOR, zorder=2)
    for _i in range(_nt):
        _ax.text(_xt[_i], _mb[_i], f"{_mb[_i]:.3f}", ha="center", va="bottom",
                 fontsize=_FONT_SIZE - 2, fontweight="bold")
    _ax.set_xticks(_xt); _ax.set_xticklabels(TIER_ORDER)
    _ax.set_xlim(-0.5, _nt - 0.5)
    _ax.set_ylim(0, _safe_top(_mb))
    _ax.set_ylabel("Brier Score")
    _panel_label(_ax, "d", "Mean Brier Score by Tier")

    _fig.suptitle(f"{CLASSIFIER_NAME} on Independent Capture Session — Preprocessing Tier Comparison")
    _fig.tight_layout()
    save_fig(_fig, "tier_summary")
    plt.show()


In [ ]:
# === Cell 9: Plot Deployment Confusion Matrices ===

_apply_rcparams()

# --- Run 1: LDPE excluded, full metric panel ---
_csv = latest_csv("predictions_DEPLOY_NO_LDPE")
if _csv is None:
    print("No DEPLOY_NO_LDPE prediction CSV found. Run the deployment batch cell first.")
else:
    _df = pd.read_csv(_csv)
    _cols = pred_axis([c[len("proba_"):] for c in _df.columns if c.startswith("proba_")])
    _rows = DEPLOY_ROW_MERGED
    _n = len(_rows)

    _cm, _ = rect_confusion(_df["true_class"], _df["pred_class"], _rows, _cols)
    _, _, _f1, _ = precision_recall_fscore_support(
        _df["true_class"], _df["pred_class"], labels=_rows, average=None, zero_division=0
    )
    _brier = per_class_brier(_df, _rows)
    _brier_vals = [_brier[c] for c in _rows]

    _fig, _axes = plt.subplots(1, 3, figsize=(_FW, _FW * 0.36))

    _ax = _axes[0]
    _draw_cm(_ax, _cm)
    _panel_label(_ax, "a", "Confusion Matrix")
    _ax.set_box_aspect(1)

    _ax = _axes[1]
    _add_bands(_ax, _n)
    _ax.bar(range(_n), _f1, color=_F1_COLOR, zorder=2)
    _ax.set_xticks(range(_n)); _ax.set_xticklabels(_rows, rotation=45, ha="right")
    _ax.set_xlim(-0.5, _n - 0.5)
    _ax.set_ylim(0, _safe_top(_f1))
    _ax.set_ylabel("F1 Score")
    _panel_label(_ax, "b", "Per-Class F1 Score")
    _ax.set_box_aspect(1)

    _ax = _axes[2]
    _add_bands(_ax, _n)
    _ax.bar(range(_n), _brier_vals, color=_BRIER_COLOR, zorder=2)
    _ax.set_xticks(range(_n)); _ax.set_xticklabels(_rows, rotation=45, ha="right")
    _ax.set_xlim(-0.5, _n - 0.5)
    _ax.set_ylim(0, _safe_top(_brier_vals))
    _ax.set_ylabel("Brier Score")
    _panel_label(_ax, "c", "Per-Class Brier Score")
    _ax.set_box_aspect(1)

    _fig.suptitle(f"{CLASSIFIER_NAME} Deployment on Independent Capture Session — LDPE Excluded")
    _fig.tight_layout()
    save_fig(_fig, "deployment_no_ldpe")
    plt.show()

# --- Run 2: LDPE carried separately, counts only ---
_csv = latest_csv("predictions_DEPLOY_LDPE_SPLIT")
if _csv is None:
    print("No DEPLOY_LDPE_SPLIT prediction CSV found. Run the deployment batch cell first.")
else:
    _df = pd.read_csv(_csv)
    _cols = pred_axis([c[len("proba_"):] for c in _df.columns if c.startswith("proba_")])
    _rows = DEPLOY_ROW_SPLIT

    _cm, _ = rect_confusion(_df["true_class"], _df["pred_class"], _rows, _cols)
    _support = _cm.sum(axis=1).astype(float).replace(0.0, np.nan)
    _cm_pct = _cm.div(_support, axis=0) * 100.0

    _annot = [[f"{int(_cm.iat[i, j])}" for j in range(_cm.shape[1])]
              for i in range(_cm.shape[0])]

    _fig, _ax = plt.subplots(1, 1, figsize=(_FW * 0.5, _FW * 0.5))
    _draw_cm(_ax, _cm, annot=_annot)
    _ax.set_box_aspect(1)

    _fig.tight_layout()
    save_fig(_fig, "deployment_ldpe_split")
    plt.show()

    print("\nRow shares by true class:")
    print(_cm_pct.round(1).to_string())